We want to explore openstreetmap data to see if we can extract some interesting features from it.
The idea is to extract schools, hospitals etc. and add the distance to the nearest (or just bool of presence) as a feature for our municipality feature matrix.
First experiment is focused on extracting hospitals from openstreetmap data.

Run following commands

```bash
# download germany data
curl https://download.geofabrik.de/europe/germany-latest.osm.pbf --output data/raw/osm/germany.pbf
```

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

from geoscore_de.data_flow.features.municipality import MunicipalityFeature
from geoscore_de.data_flow.geo import load_geo_data

muni = (
    MunicipalityFeature(raw_data_path="../../data/raw/municipalities_2022.csv")
    .load_transform()
    .drop(columns=["federal_state_id", "admin_region_id"])
)

gdf_muni = load_geo_data("../../data/georef-germany-gemeinde.csv")

gdf_muni = gdf_muni.merge(muni, on="AGS", how="inner")

## Hospitals in Germany

```bash
# extract hospitals
osmium tags-filter data/osm/germany.osm.pbf amenity=hospital -o data/osm/hospitals.osm.pbf

# convert to geojson
osmium export data/osm/hospitals.osm.pbf -o data/osm/hospitals.geojson
```


Number of hospitals in Germany should be something around 2000 based on statista website: https://www.statista.com/statistics/578444/number-of-hospitals-germany/?srsltid=AfmBOoo6c_JS-0Vk-aeegnn0Eoe5HibTJx7NNOZHW8zXdyXD7QKLOYty

So we need to filter out some of the hospitals. We can do this by filtering out hospitals that are not in Germany. We can use the "addr:country" tag to filter out hospitals that are not officially in Germany.

In [ ]:
gdf_hospital = gpd.read_file("../../data/raw/osm/hospitals.geojson")
gdf_hospital

In [ ]:
gdf_hospital_de = gdf_hospital[gdf_hospital["addr:country"] == "DE"][["name", "geometry"]]
gdf_hospital_de

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
gdf_hospital_de.plot(ax=ax, markersize=10)
gdf_muni.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.2)
plt.title("Hospitals in Germany")
plt.show()

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

# Load municipality centroids
gdf_muni["centroid"] = gdf_muni.geometry.centroid
mun_coords = np.radians(np.column_stack([gdf_muni.centroid.y, gdf_muni.centroid.x]))

hosp_coords = np.radians(np.column_stack([gdf_hospital_de.geometry.centroid.y, gdf_hospital_de.geometry.centroid.x]))

# Build tree
tree = BallTree(hosp_coords, metric="haversine")

# Nearest 1
dist, _ = tree.query(mun_coords, k=1)
gdf_muni["dist_nearest_hospital_km"] = dist[:, 0] * 6371  # earth radius

# Nearest 3 (mean)
dist3, _ = tree.query(mun_coords, k=3)
gdf_muni["dist_mean_3_hospitals_km"] = dist3.mean(axis=1) * 6371

# Count within 10km
radius_km = 10
idx = tree.query_radius(mun_coords, r=radius_km / 6371)
gdf_muni["hospital_count_10km"] = [len(i) for i in idx]

# Count within 30km
radius_km = 30
idx = tree.query_radius(mun_coords, r=radius_km / 6371)
gdf_muni["hospital_count_30km"] = [len(i) for i in idx]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="dist_nearest_hospital_km", cmap="viridis_r", legend=True, vmin=0)
plt.title("Distance to Nearest Hospital (km)")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="dist_mean_3_hospitals_km", cmap="viridis_r", legend=True, vmin=0)
plt.title("Distance to Mean of 3 Nearest Hospitals (km)")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="hospital_count_30km", cmap="viridis", legend=True)
plt.title("Count of Hospitals within 30km Radius from Municipality Centroid")
plt.show()

The feature with distance to 3 nearest hospitals from the center of the municipality looks really promising.
It is smooth version of the distance to the nearest hospital and it is not affected by outliers (like a single hospital that is very close to the municipality but there are no other hospitals nearby).

## ATM

In [ ]:
# !osmium tags-filter ../../data/raw/osm/germany.osm.pbf amenity=atm -o ../../data/raw/osm/atms.osm.pbf
# !osmium export ../../data/raw/osm/atms.osm.pbf -o ../../data/raw/osm/atms.geojson

In [ ]:
# gdf_atm = gpd.read_file("../../data/raw/osm/atms.geojson")
# gdf_atm.shape

```bash
!osmium tags-filter ../../data/raw/osm/germany.osm.pbf \
  amenity=atm \
  atm=yes \
  atm=only \
  amenity=bank \
  -o ../../data/raw/osm/atms_full.osm.pbf \
  --overwrite

!osmium export ../../data/raw/osm/atms_full.osm.pbf -o ../../data/raw/osm/atms_full.geojson --overwrite
```

In [ ]:
gdf_atm = gpd.read_file("../../data/raw/osm/atms_full.geojson")
gdf_atm.shape

According to statista there should be around 75k ATMs in Germany: https://www.statista.com/statistics/473556/number-of-cash-machines-germany/?srsltid=AfmBOoqBiTewpqXrmrz6Dfg7kq8_Fn0oJCiqvbdwzfc9F6zXW-fQb1_q
Our data contain only 40k even after counting all banks as ATMs as well. 

In [ ]:
print(gdf_atm["amenity"].value_counts())

In [ ]:
gdf_atm["geometry"] = gdf_atm.geometry.centroid

# --- Within borders ---
joined = gpd.sjoin(gdf_atm, gdf_muni, how="left", predicate="within")
counts = joined.groupby("AGS").size().reset_index(name="atm_count_within")
muns = gdf_muni.merge(counts, on="AGS", how="left").fillna(0)

In [ ]:
# --- Per capita ---
muns["atms_per_1000_residents"] = (muns["atm_count_within"] / muns["Persons"]) * 1000
muns["atm_density_per_km2"] = muns["atm_count_within"] / muns["Area"]

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

muns.plot(ax=ax, column="atms_per_1000_residents", cmap="viridis", legend=True)
plt.title("ATMs per 1000 Residents")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

muns.plot(ax=ax, column="atm_density_per_km2", cmap="viridis", legend=True)
plt.title("ATM Density per km²")
plt.show()

## Tourist attractions

In [ ]:
!osmium tags-filter ../../data/raw/osm/germany.osm.pbf \
  n/tourism=attraction \
  n/tourism=museum \
  n/tourism=viewpoint \
  n/historic=castle \
  n/historic=monument \
  n/historic=ruins \
  n/historic=archaeological_site \
  n/historic=memorial \
  -o ../../data/raw/osm/tourism.osm.pbf \
  --overwrite

In [ ]:
!osmium export ../../data/raw/osm/tourism.osm.pbf -o ../../data/raw/osm/tourism.geojson --overwrite

In [ ]:
import geopandas as gpd

gdf_tourist = gpd.read_file("../../data/raw/osm/tourism.geojson")
gdf_tourist.shape

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_tourist.plot(ax=ax, markersize=0.2)

In [ ]:
# --- POI count within municipality borders ---
joined = gpd.sjoin(gdf_tourist, gdf_muni, how="left", predicate="within")
counts = joined.groupby("AGS").size().reset_index(name="poi_count")
gdf_muni = gdf_muni.merge(counts, on="AGS", how="left").fillna(0)

In [ ]:
# --- Feature engineering ---
gdf_muni["poi_per_km2"] = gdf_muni["poi_count"] / gdf_muni["Area"]
gdf_muni["poi_per_1000"] = gdf_muni["poi_count"] / gdf_muni["Persons"] * 1000

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

# Load municipality centroids
gdf_muni["centroid"] = gdf_muni.geometry.centroid
mun_coords = np.radians(np.column_stack([gdf_muni.centroid.y, gdf_muni.centroid.x]))

tourism_coords = np.radians(np.column_stack([gdf_tourist.geometry.centroid.y, gdf_tourist.geometry.centroid.x]))

# Build tree
tree = BallTree(tourism_coords, metric="haversine")

# Nearest 30 (mean)
dist3, _ = tree.query(mun_coords, k=30)
gdf_muni["dist_mean_30_tourism_km"] = dist3.mean(axis=1) * 6371

# Count within 10km
radius_km = 10
idx = tree.query_radius(mun_coords, r=radius_km / 6371)
gdf_muni["tourism_count_10km"] = [len(i) for i in idx]

# Count within 30km
radius_km = 30
idx = tree.query_radius(mun_coords, r=radius_km / 6371)
gdf_muni["tourism_count_30km"] = [len(i) for i in idx]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="poi_per_1000", cmap="viridis", legend=True, vmin=0, vmax=50)
plt.title("Number of Tourist Attractions/Historical Sites per 1000 Residents")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="poi_per_km2", cmap="viridis", legend=True, vmin=0, vmax=5)
plt.title("Number of Tourist Attractions/Historical Sites per km²")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="dist_mean_30_tourism_km", cmap="viridis_r", legend=True, vmin=0)
plt.title("Tourist Attractions/Historical Sites within 10km per 1000 Residents")
plt.show()

## Schools in Germany

In [ ]:
# extract universities
!osmium tags-filter ../../data/raw/osm/germany.osm.pbf \
  education=university \
  -o ../../data/raw/osm/universities.osm.pbf --overwrite
# convert to geojson
!osmium export ../../data/raw/osm/universities.osm.pbf -o ../../data/raw/osm/universities.geojson --overwrite

In [ ]:
import geopandas as gpd

gdf_uni = gpd.read_file("../../data/raw/osm/universities.geojson")
gdf_uni

In [ ]:
gdf_uni = gdf_uni[gdf_uni["name"].notna()]
gdf_uni = gdf_uni.drop_duplicates(subset="name")
gdf_uni[["name"]]

According to CHE, there should be around 109 universities and around 422 institutes of higher education in Germany: https://hochschuldaten.che.de/data-on-higher-education-in-germany/institutions/#:~:text=According%20to%20the%20German%20Federal,the%20winter%20semester%202024%2F25.
We did get 299 schools from OSM data.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_uni.plot(markersize=5, ax=ax)
gdf_muni.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.2)
plt.title("Universities in Germany")
plt.show()

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

# Load municipality centroids
gdf_muni["centroid"] = gdf_muni.geometry.centroid
mun_coords = np.radians(np.column_stack([gdf_muni.centroid.y, gdf_muni.centroid.x]))

# build university tree
uni_coords = np.radians(np.column_stack([gdf_uni.geometry.centroid.y, gdf_uni.geometry.centroid.x]))

# Build tree
tree = BallTree(uni_coords, metric="haversine")

# Nearest 1
dist, _ = tree.query(mun_coords, k=1)
gdf_muni["dist_nearest_university_km"] = dist[:, 0] * 6371  # earth radius

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

gdf_muni.plot(ax=ax, column="dist_nearest_university_km", cmap="viridis_r", legend=True, vmin=0)
plt.title("Distance to Nearest University (km)")
plt.show()